# 03 Data Preparation

ขั้นตอนนี้สร้าง target และ features สำหรับกลุ่มเชื้อเพลิง (`100`), ปุ๋ย (`318`) และนม (`402`) โดยยึด forecast origin เป็นวันที่ 25 ของเดือน `t+1`

ณ forecast origin ระบบใช้ IMI ได้ถึงเดือน `t` และใช้ DEXTHUS/Brent ได้เฉพาะ observation วันที่ 1–24 ของเดือน `t+1` สำหรับ partial-month feature มีชุด sensitivity ที่ตัดข้อมูล ณ วันที่ 17 แยกไว้ด้วย

## 1. โหลดและจัดรูปแบบข้อมูล

อ่าน snapshot ล่าสุดจาก `sources.json`, ตรวจ checksum และแปลงปี พ.ศ. ของ IMI เป็นเดือน ค.ศ. โดยไม่แก้ไขไฟล์ raw

In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'PROJECT_SCOPE.md').exists()
)
RAW_DIR = ROOT / 'data' / 'raw'
INTERIM_DIR = ROOT / 'data' / 'interim'
PROCESSED_DIR = ROOT / 'data' / 'processed'
METRICS_DIR = ROOT / 'logs' / 'metrics'
for directory in (INTERIM_DIR, PROCESSED_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

GROUPS = {'100': 'Fuel', '318': 'Fertilizer', '402': 'Dairy products'}

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

sources = json.loads((RAW_DIR / 'sources.json').read_text(encoding='utf-8'))['sources']

def latest_source(prefix: str) -> dict:
    return next(item for item in reversed(sources) if item['source_id'].startswith(prefix))

source_items = {
    'IMI': latest_source('imi-monthly-'),
    'DEXTHUS': latest_source('dexthus-'),
    'DCOILBRENTEU': latest_source('dcoilbrenteu-'),
}
for item in source_items.values():
    path = RAW_DIR / item['file']
    assert sha256(path) == item['sha256'], f'checksum ไม่ตรง: {path.name}'

imi_raw = pd.read_csv(
    RAW_DIR / source_items['IMI']['file'],
    dtype={'commodityCode': 'string'},
)
imi_raw['commodityCode'] = imi_raw['commodityCode'].str.zfill(3)
imi_raw['month_ce'] = pd.to_datetime(dict(
    year=imi_raw['year'] - 543, month=imi_raw['month'], day=1,
))
imi = (
    imi_raw[imi_raw['commodityCode'].isin(GROUPS)]
    .loc[:, ['commodityCode', 'commodityNameTH', 'month_ce', 'index']]
    .rename(columns={
        'commodityCode': 'commodity_code',
        'commodityNameTH': 'commodity_name_th',
        'index': 'imi_index',
    })
    .sort_values(['commodity_code', 'month_ce'])
    .reset_index(drop=True)
)
assert len(imi) == 3 * 320
assert not imi.duplicated(['commodity_code', 'month_ce']).any()
assert (imi['imi_index'] > 0).all()
display(imi.groupby('commodity_code').agg(
    rows=('imi_index', 'size'),
    start=('month_ce', 'min'),
    end=('month_ce', 'max'),
))

,rows,start,end
commodity_code,,,
100,320,2000-01-01,2026-08-01
318,320,2000-01-01,2026-08-01
402,320,2000-01-01,2026-08-01


## 2. สร้างตารางข้อมูลภายนอกรายเดือน

สำหรับแต่ละเดือนคำนวณค่าเฉลี่ยทั้งเดือน ค่าเฉลี่ยวันที่ 1–24 และค่าเฉลี่ยวันที่ 1–17 โดย `mean()` ข้ามค่าว่างและไม่แทนค่าที่หายด้วยศูนย์ พร้อมเก็บจำนวน observation และวันสุดท้ายที่ใช้เพื่อ audit leakage

In [2]:
external_parts = []
for series_id in ('DEXTHUS', 'DCOILBRENTEU'):
    frame = pd.read_csv(
        RAW_DIR / source_items[series_id]['file'],
        na_values=['.'],
    )
    frame['observation_date'] = pd.to_datetime(frame['observation_date'], errors='raise')
    frame = frame[frame['observation_date'] >= '2000-01-01'].copy()
    frame['month_ce'] = frame['observation_date'].dt.to_period('M').dt.to_timestamp()
    frame['day'] = frame['observation_date'].dt.day

    full = frame.groupby('month_ce').agg(
        full_mean=(series_id, 'mean'),
        full_n=(series_id, 'count'),
    )
    through_24 = frame[frame['day'] <= 24].groupby('month_ce').agg(
        partial_24_mean=(series_id, 'mean'),
        partial_24_n=(series_id, 'count'),
        partial_24_max_date=('observation_date', lambda x: x[frame.loc[x.index, series_id].notna()].max()),
    )
    through_17 = frame[frame['day'] <= 17].groupby('month_ce').agg(
        partial_17_mean=(series_id, 'mean'),
        partial_17_n=(series_id, 'count'),
        partial_17_max_date=('observation_date', lambda x: x[frame.loc[x.index, series_id].notna()].max()),
    )
    monthly = full.join(through_24).join(through_17).reset_index()
    monthly.insert(0, 'series', series_id)
    external_parts.append(monthly)

external_monthly_long = pd.concat(external_parts, ignore_index=True)
external_monthly_long.to_csv(
    INTERIM_DIR / 'external_monthly_aggregates.csv', index=False,
)
display(external_monthly_long.groupby('series').agg(
    months=('month_ce', 'size'),
    min_n_24=('partial_24_n', 'min'),
    min_n_17=('partial_17_n', 'min'),
    start=('month_ce', 'min'),
    end=('month_ce', 'max'),
))

,months,min_n_24,min_n_17,start,end
series,,,,,
DCOILBRENTEU,321,7,4,2000-01-01,2026-09-01
DEXTHUS,321,13,9,2000-01-01,2026-09-01


## 3. สร้าง features ณ เดือน `t`

IMI features ใช้เฉพาะค่าถึงเดือน `t` ส่วน partial-month feature เปรียบเทียบค่าเฉลี่ยวันที่ 1–24 หรือ 1–17 ของเดือน `t+1` กับค่าเฉลี่ยทั้งเดือน `t` Rolling volatility ใช้ส่วนเบี่ยงเบนมาตรฐานย้อนหลัง 6 ค่าของ log change 1 เดือน

In [3]:
feature_frames = []
for code, group in imi.groupby('commodity_code', sort=False):
    group = group.copy().sort_values('month_ce')
    log_index = np.log(group['imi_index'])
    for window in (1, 2, 3, 6, 12):
        group[f'imi_log_change_{window}m_pct'] = 100 * (log_index - log_index.shift(window))
    group['imi_volatility_6m_pct'] = (
        group['imi_log_change_1m_pct'].rolling(6, min_periods=6).std(ddof=1)
    )
    feature_frames.append(group)
base_features = pd.concat(feature_frames, ignore_index=True)

for series_id, prefix in (('DEXTHUS', 'fx'), ('DCOILBRENTEU', 'brent')):
    monthly = (
        external_monthly_long[external_monthly_long['series'] == series_id]
        .copy()
        .sort_values('month_ce')
        .set_index('month_ce')
    )
    log_mean = np.log(monthly['full_mean'])
    history = pd.DataFrame(index=monthly.index)
    for window in (1, 2, 3):
        history[f'{prefix}_log_change_{window}m_pct'] = 100 * (log_mean - log_mean.shift(window))
    history[f'{prefix}_volatility_6m_pct'] = (
        (100 * log_mean.diff()).rolling(6, min_periods=6).std(ddof=1)
    )
    history = history.reset_index()
    base_features = base_features.merge(history, on='month_ce', how='left', validate='many_to_one')

    future_partial = monthly[[
        'partial_24_mean', 'partial_24_n', 'partial_24_max_date',
        'partial_17_mean', 'partial_17_n', 'partial_17_max_date',
    ]].copy()
    future_partial.index = future_partial.index - pd.offsets.MonthBegin(1)
    future_partial.index.name = 'month_ce'
    future_partial[f'{prefix}_partial_24_pct'] = 100 * np.log(
        future_partial['partial_24_mean'] / monthly['full_mean']
    )
    future_partial[f'{prefix}_partial_17_pct'] = 100 * np.log(
        future_partial['partial_17_mean'] / monthly['full_mean']
    )
    future_partial = future_partial.rename(columns={
        'partial_24_n': f'{prefix}_partial_24_n',
        'partial_24_max_date': f'{prefix}_partial_24_max_date',
        'partial_17_n': f'{prefix}_partial_17_n',
        'partial_17_max_date': f'{prefix}_partial_17_max_date',
    })
    keep = [
        f'{prefix}_partial_24_pct', f'{prefix}_partial_17_pct',
        f'{prefix}_partial_24_n', f'{prefix}_partial_17_n',
        f'{prefix}_partial_24_max_date', f'{prefix}_partial_17_max_date',
    ]
    base_features = base_features.merge(
        future_partial[keep].reset_index(),
        on='month_ce', how='left', validate='many_to_one',
    )

base_features['forecast_origin'] = base_features['month_ce'] + pd.offsets.MonthBegin(1) + pd.Timedelta(days=24)
display(base_features.tail(3).T)

,957,958,959
commodity_code,402,402,402
commodity_name_th,4.2 นมและผลิตภัณฑ์นม,4.2 นมและผลิตภัณฑ์นม,4.2 นมและผลิตภัณฑ์นม
month_ce,2026-06-01 00:00:00,2026-07-01 00:00:00,2026-08-01 00:00:00
imi_index,135.4,134.2,136.2
imi_log_change_1m_pct,-0.809125,-0.890214,1.479317
imi_log_change_2m_pct,-1.757,-1.699339,0.589103
imi_log_change_3m_pct,-3.126391,-2.647213,-0.220022
imi_log_change_6m_pct,-3.83966,-3.729866,-1.312929
imi_log_change_12m_pct,-12.35109,-11.99334,-9.3172
imi_volatility_6m_pct,0.932675,0.925245,1.234585


## 4. สร้าง target สำหรับ `h=1` และ `h=2`

Target คือ `100 × [log(IMI[t+h]) − log(IMI[t])]` ตารางใช้รูปแบบ long โดยหนึ่งแถวต่อกลุ่ม เดือน `t` และ horizon พร้อมบันทึกเดือนเป้าหมายและวันที่ประมาณการว่า target จะเผยแพร่

In [4]:
prepared_parts = []
for horizon in (1, 2):
    horizon_frames = []
    for code, group in base_features.groupby('commodity_code', sort=False):
        group = group.copy().sort_values('month_ce')
        group['horizon'] = horizon
        group['target_month'] = group['month_ce'] + pd.offsets.MonthBegin(horizon)
        group['target_log_change_pct'] = 100 * (
            np.log(group['imi_index'].shift(-horizon)) - np.log(group['imi_index'])
        )
        group['target_available_date'] = (
            group['target_month'] + pd.offsets.MonthBegin(1) + pd.Timedelta(days=24)
        )
        horizon_frames.append(group)
    prepared_parts.append(pd.concat(horizon_frames, ignore_index=True))
prepared = pd.concat(prepared_parts, ignore_index=True)

IMI_FEATURES = [
    'imi_log_change_1m_pct', 'imi_log_change_2m_pct',
    'imi_log_change_3m_pct', 'imi_log_change_6m_pct',
    'imi_log_change_12m_pct', 'imi_volatility_6m_pct',
]
FX_HISTORY = [
    'fx_log_change_1m_pct', 'fx_log_change_2m_pct',
    'fx_log_change_3m_pct', 'fx_volatility_6m_pct',
]
BRENT_HISTORY = [
    'brent_log_change_1m_pct', 'brent_log_change_2m_pct',
    'brent_log_change_3m_pct', 'brent_volatility_6m_pct',
]
PARTIAL_24 = ['fx_partial_24_pct', 'brent_partial_24_pct']
PARTIAL_17 = ['fx_partial_17_pct', 'brent_partial_17_pct']
ALL_REQUIRED = IMI_FEATURES + FX_HISTORY + BRENT_HISTORY + PARTIAL_24 + PARTIAL_17

prepared = prepared.dropna(subset=ALL_REQUIRED + ['target_log_change_pct']).copy()
prepared['evaluation_split'] = np.select(
    [
        prepared['month_ce'] <= '2014-12-01',
        prepared['month_ce'] <= '2018-12-01',
    ],
    ['train_initial', 'validation'],
    default='test',
)
prepared = prepared.sort_values(['commodity_code', 'horizon', 'month_ce']).reset_index(drop=True)

processed_path = PROCESSED_DIR / 'modeling_table.csv'
prepared.to_csv(processed_path, index=False)
processed_manifest = pd.DataFrame([{
    'file': processed_path.name,
    'rows': len(prepared),
    'columns': len(prepared.columns),
    'sha256': sha256(processed_path),
}])
processed_manifest.to_csv(METRICS_DIR / '03_processed_manifest.csv', index=False)
row_counts = (
    prepared.groupby(['commodity_code', 'horizon', 'evaluation_split'])
    .size().rename('rows').reset_index()
)
row_counts.to_csv(METRICS_DIR / '03_processed_row_counts.csv', index=False)
display(row_counts)
print(f'บันทึก {len(prepared):,} แถว: {processed_path}')

,commodity_code,horizon,evaluation_split,rows
0,100,1,test,91
1,100,1,train_initial,168
2,100,1,validation,48
3,100,2,test,90
4,100,2,train_initial,168
5,100,2,validation,48
6,318,1,test,91
7,318,1,train_initial,168
8,318,1,validation,48
9,318,2,test,90


บันทึก 1,839 แถว: /Users/pattarapol/itkmitl/imi-early-warning/data/processed/modeling_table.csv


## 5. กำหนด feature sets ล่วงหน้า

บันทึกรายชื่อ feature อย่างชัดเจนเพื่อป้องกันการเลือก feature จากผลบน test set ชุด `3-day17` ใช้เฉพาะ sensitivity check

In [5]:
FEATURE_SETS = {
    '1': IMI_FEATURES,
    '2': IMI_FEATURES + FX_HISTORY + ['fx_partial_24_pct'],
    '3': IMI_FEATURES + FX_HISTORY + BRENT_HISTORY + PARTIAL_24,
    '3-noPartial': IMI_FEATURES + FX_HISTORY + BRENT_HISTORY,
    '3-day17': IMI_FEATURES + FX_HISTORY + BRENT_HISTORY + PARTIAL_17,
}
feature_set_rows = [
    {'feature_set': set_name, 'feature_order': order, 'feature': feature}
    for set_name, features in FEATURE_SETS.items()
    for order, feature in enumerate(features, start=1)
]
feature_set_manifest = pd.DataFrame(feature_set_rows)
feature_set_manifest.to_csv(METRICS_DIR / '03_feature_sets.csv', index=False)
display(feature_set_manifest.groupby('feature_set').agg(
    number_of_features=('feature', 'size'),
    features=('feature', lambda values: ', '.join(values)),
))

,number_of_features,features
feature_set,,
1,6,"imi_log_change_1m_pct, imi_log_change_2m_pct, ..."
2,11,"imi_log_change_1m_pct, imi_log_change_2m_pct, ..."
3,16,"imi_log_change_1m_pct, imi_log_change_2m_pct, ..."
3-day17,16,"imi_log_change_1m_pct, imi_log_change_2m_pct, ..."
3-noPartial,14,"imi_log_change_1m_pct, imi_log_change_2m_pct, ..."


## 6. Leakage audit และ unit checks

ตรวจทุกแถวว่า partial feature ใช้ observation ไม่เกินวันที่ 24 หรือ 17 ของเดือน `t+1`, target อยู่หลัง forecast origin, สูตร target ถูกต้อง และไม่มี feature หลักที่เป็นค่าว่าง

In [6]:
audit_results = []

for prefix in ('fx', 'brent'):
    max_24 = pd.to_datetime(prepared[f'{prefix}_partial_24_max_date'])
    max_17 = pd.to_datetime(prepared[f'{prefix}_partial_17_max_date'])
    cutoff_24 = prepared['forecast_origin'] - pd.Timedelta(days=1)
    cutoff_17 = prepared['forecast_origin'] - pd.Timedelta(days=8)
    audit_results.append({
        'check': f'{prefix}_partial_24_max_date <= day 24',
        'rows_checked': len(prepared),
        'violations': int((max_24 > cutoff_24).sum()),
    })
    audit_results.append({
        'check': f'{prefix}_partial_17_max_date <= day 17',
        'rows_checked': len(prepared),
        'violations': int((max_17 > cutoff_17).sum()),
    })

audit_results.extend([
    {
        'check': 'target_month > month_t',
        'rows_checked': len(prepared),
        'violations': int((prepared['target_month'] <= prepared['month_ce']).sum()),
    },
    {
        'check': 'target_available_date > forecast_origin',
        'rows_checked': len(prepared),
        'violations': int((prepared['target_available_date'] <= prepared['forecast_origin']).sum()),
    },
    {
        'check': 'required_features_are_not_missing',
        'rows_checked': len(prepared),
        'violations': int(prepared[ALL_REQUIRED].isna().any(axis=1).sum()),
    },
])

lookup = imi.set_index(['commodity_code', 'month_ce'])['imi_index']
recomputed_target = prepared.apply(
    lambda row: 100 * (
        np.log(lookup.loc[(row['commodity_code'], row['target_month'])])
        - np.log(row['imi_index'])
    ),
    axis=1,
)
target_mismatch = ~np.isclose(
    prepared['target_log_change_pct'], recomputed_target, rtol=0, atol=1e-12,
)
audit_results.append({
    'check': 'target_formula_matches_source_index',
    'rows_checked': len(prepared),
    'violations': int(target_mismatch.sum()),
})

leakage_audit = pd.DataFrame(audit_results)
leakage_audit.to_csv(METRICS_DIR / '03_leakage_audit.csv', index=False)
display(leakage_audit)
assert leakage_audit['violations'].sum() == 0

sample_columns = [
    'commodity_code', 'month_ce', 'forecast_origin', 'horizon',
    'target_month', 'target_available_date',
    'fx_partial_24_max_date', 'brent_partial_24_max_date',
    'target_log_change_pct',
]
audit_sample = (
    prepared.groupby(['commodity_code', 'horizon'], group_keys=False)
    .tail(1)[sample_columns]
    .reset_index(drop=True)
)
audit_sample.to_csv(METRICS_DIR / '03_leakage_audit_sample.csv', index=False)
display(audit_sample)

,check,rows_checked,violations
0,fx_partial_24_max_date <= day 24,1839,0
1,fx_partial_17_max_date <= day 17,1839,0
2,brent_partial_24_max_date <= day 24,1839,0
3,brent_partial_17_max_date <= day 17,1839,0
4,target_month > month_t,1839,0
5,target_available_date > forecast_origin,1839,0
6,required_features_are_not_missing,1839,0
7,target_formula_matches_source_index,1839,0


,commodity_code,month_ce,forecast_origin,horizon,target_month,target_available_date,fx_partial_24_max_date,brent_partial_24_max_date,target_log_change_pct
0,100,2026-07-01,2026-08-25,1,2026-08-01,2026-09-25,2026-08-24,2026-08-24,3.416481
1,100,2026-06-01,2026-07-25,2,2026-08-01,2026-09-25,2026-07-24,2026-07-24,-0.608830
2,318,2026-07-01,2026-08-25,1,2026-08-01,2026-09-25,2026-08-24,2026-08-24,-1.910886
3,318,2026-06-01,2026-07-25,2,2026-08-01,2026-09-25,2026-07-24,2026-07-24,-7.234559
4,402,2026-07-01,2026-08-25,1,2026-08-01,2026-09-25,2026-08-24,2026-08-24,1.479317
5,402,2026-06-01,2026-07-25,2,2026-08-01,2026-09-25,2026-07-24,2026-07-24,0.589103


## 7. สรุปและ checkpoint

จำนวนแถวถูกกำหนดจาก lag สูงสุด 12 เดือนและ target ที่ต้องมีค่าจริง ปลายชุดข้อมูลจึงมีแถว `h=1` มากกว่า `h=2` กลุ่มละหนึ่งแถว

In [7]:
summary = (
    prepared.groupby(['commodity_code', 'horizon'])
    .agg(
        rows=('target_log_change_pct', 'size'),
        first_t=('month_ce', 'min'),
        last_t=('month_ce', 'max'),
        first_target=('target_month', 'min'),
        last_target=('target_month', 'max'),
    )
    .reset_index()
)
display(summary)

h1_counts = summary[summary['horizon'] == 1].set_index('commodity_code')['rows']
h2_counts = summary[summary['horizon'] == 2].set_index('commodity_code')['rows']
assert (h1_counts == 307).all()
assert (h2_counts == 306).all()
assert len(prepared.query("evaluation_split == 'test' and horizon == 2")) == 270
assert processed_path.exists() and processed_path.stat().st_size > 0
assert sha256(processed_path) == processed_manifest.loc[0, 'sha256']
assert not prepared.duplicated(['commodity_code', 'month_ce', 'horizon']).any()

display(Markdown(f'''
- ตาราง processed มี **{len(prepared):,} แถว** และ {len(prepared.columns)} คอลัมน์
- `h=1` มี 307 แถวต่อกลุ่ม และ `h=2` มี 306 แถวต่อกลุ่ม
- ช่วง test ของ `h=2` มี **90 forecast origins ต่อกลุ่ม** ตั้งแต่ 2019-01 ถึง 2026-06
- Leakage audit ตรวจ {len(prepared):,} แถว และไม่พบ violation
- วันที่เผยแพร่ IMI ใช้สมมติฐานวันที่ 25 ของเดือนถัดไปตาม Scope เนื่องจากวันเผยแพร่จริงย้อนหลังยังไม่ยืนยัน
'''))
print('CHECKPOINT PASSED: ตาราง feature/target พร้อมใช้และไม่พบการใช้ข้อมูลหลัง cutoff ที่กำหนด')

,commodity_code,horizon,rows,first_t,last_t,first_target,last_target
0,100,1,307,2001-01-01,2026-07-01,2001-02-01,2026-08-01
1,100,2,306,2001-01-01,2026-06-01,2001-03-01,2026-08-01
2,318,1,307,2001-01-01,2026-07-01,2001-02-01,2026-08-01
3,318,2,306,2001-01-01,2026-06-01,2001-03-01,2026-08-01
4,402,1,307,2001-01-01,2026-07-01,2001-02-01,2026-08-01
5,402,2,306,2001-01-01,2026-06-01,2001-03-01,2026-08-01



- ตาราง processed มี **1,839 แถว** และ 36 คอลัมน์
- `h=1` มี 307 แถวต่อกลุ่ม และ `h=2` มี 306 แถวต่อกลุ่ม
- ช่วง test ของ `h=2` มี **90 forecast origins ต่อกลุ่ม** ตั้งแต่ 2019-01 ถึง 2026-06
- Leakage audit ตรวจ 1,839 แถว และไม่พบ violation
- วันที่เผยแพร่ IMI ใช้สมมติฐานวันที่ 25 ของเดือนถัดไปตาม Scope เนื่องจากวันเผยแพร่จริงย้อนหลังยังไม่ยืนยัน


CHECKPOINT PASSED: ตาราง feature/target พร้อมใช้และไม่พบการใช้ข้อมูลหลัง cutoff ที่กำหนด


In [8]:
process_log_path = ROOT / 'logs' / 'process_log.md'
existing_log = process_log_path.read_text(encoding='utf-8')
marker = 'ขั้นตอน: 03 Data Preparation'
if marker not in existing_log:
    entry = f'''

## [2026-10-05] ขั้นตอน: 03 Data Preparation
**สิ่งที่ทำ:** แปลงเดือน IMI จาก พ.ศ. เป็น ค.ศ. สร้าง target h=1 และ h=2 สร้าง IMI history, DEXTHUS, Brent, partial-month วันที่ 24 และ sensitivity วันที่ 17 พร้อมกำหนด feature sets และตรวจ data leakage รายแถว

**ผลที่ได้:** ตาราง processed มี {len(prepared):,} แถว h=1 จำนวน 307 แถวต่อกลุ่ม และ h=2 จำนวน 306 แถวต่อกลุ่ม ช่วง test ของ h=2 มี 90 origins ต่อกลุ่ม Leakage audit ไม่พบ violation จาก {len(leakage_audit)} เงื่อนไขตรวจสอบ

**สิ่งที่พบ / ปัญหา:** วันเผยแพร่ IMI จริงย้อนหลังยังไม่ยืนยัน จึงใช้สมมติฐาน forecast origin วันที่ 25 ตาม Scope ข้อมูล FRED ใช้ observation date เป็นตัวแทน availability และต้องประเมิน sensitivity วันที่ 17 เทียบวันที่ 24 ในขั้น Evaluation

**การตัดสินใจและเหตุผล:** เก็บตารางแบบ long แยก horizon เพื่อให้ rolling-origin training ตรวจ target availability ได้โดยตรง บันทึก target_available_date และวัน observation ล่าสุดของ partial features เพื่อ audit ย้อนกลับ

**ขั้นต่อไป:** ดำเนินการ 04 Modeling โดยตัด training rows ที่ target_available_date อยู่หลัง forecast origin ของแต่ละรอบ และ fit preprocessing เฉพาะ training data
'''
    with process_log_path.open('a', encoding='utf-8') as handle:
        handle.write(entry)
    print('บันทึกผลลง logs/process_log.md แล้ว')
else:
    print('มีรายการ 03 Data Preparation ใน process log แล้ว จึงไม่บันทึกซ้ำ')

มีรายการ 03 Data Preparation ใน process log แล้ว จึงไม่บันทึกซ้ำ
